# DQX Basics on Databricks Free Edition

This notebook introduces **DQX (Data Quality eXtended)**, an open-source Databricks Labs framework for checking PySpark DataFrames. Everything uses in-memory data, so no cloud storage, catalog setup, secrets, or paid feature is required.

You will learn how to:

- define row-level and dataset-level checks;
- distinguish `error` rules from `warn` rules;
- attach detailed quality results to rows;
- split valid and quarantined records; and
- create a simple pipeline quality gate.

> Attach this Python notebook to Databricks Free Edition serverless compute and run it from top to bottom. DQX requires Spark 3.5 or newer.

## 1. Install DQX

For notebook-based checks, install DQX as a standalone library. The full DQX workspace tool and DQX Studio are outside this beginner lesson.

In [ ]:
%pip install databricks-labs-dqx

## 2. Import DQX and check Spark

`WorkspaceClient()` automatically uses the identity of the current Databricks notebook. No token should be placed in notebook code.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQDatasetRule, DQRowRule
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Spark version:", spark.version)
major, minor = map(int, spark.version.split(".")[:2])
assert (major, minor) >= (3, 5), "DQX requires Spark 3.5 or newer"

## 3. DQX mental model

DQX turns data-quality rules into Spark expressions and evaluates the rules together.

| Concept | Meaning |
|---|---|
| Row rule | checks each record, such as `amount` being in range |
| Dataset rule | uses information across records, such as uniqueness |
| `error` criticality | invalid record is sent only to quarantine |
| `warn` criticality | record appears in valid and quarantine outputs |
| `_errors` / `_warnings` | structured details explaining failed checks |
| DQEngine | validates definitions and applies checks to Spark data |

DQX does not silently repair bad values. It identifies them so a pipeline can accept, quarantine, report, or remediate them deliberately.

## 4. Create sample orders

The input intentionally contains:

- a blank customer ID;
- a negative amount;
- an unknown status;
- a duplicate order ID; and
- one high-value order that should produce only a warning.

In [ ]:
order_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), nullable=False),
    T.StructField("customer_id", T.StringType(), nullable=True),
    T.StructField("amount", T.DoubleType(), nullable=False),
    T.StructField("status", T.StringType(), nullable=False),
])

orders = [
    (101, "C001", 120.50, "PAID"),
    (102, "C002", 75.00, "PENDING"),
    (103, "C003", 1800.00, "PAID"),       # warning only
    (104, "", 49.99, "CANCELLED"),       # blank customer
    (105, "C005", -25.00, "PAID"),       # negative amount
    (106, "C006", 60.00, "UNKNOWN"),     # invalid status
    (106, "C007", 80.00, "PAID"),        # duplicate ID
]

orders_df = spark.createDataFrame(orders, order_schema)
display(orders_df)

## 5. Define quality checks with Python classes

Class-based rules provide editor autocomplete and validate their arguments when constructed. Give each rule a meaningful, stable name because that name appears in results and monitoring.

In [ ]:
checks = [
    DQRowRule(
        name="customer_id_required",
        criticality="error",
        check_func=check_funcs.is_not_null_and_not_empty,
        column="customer_id",
        check_func_kwargs={"trim_strings": True},
    ),
    DQRowRule(
        name="amount_valid_range",
        criticality="error",
        check_func=check_funcs.is_in_range,
        column="amount",
        check_func_kwargs={"min_limit": 0, "max_limit": 10_000},
    ),
    DQRowRule(
        name="status_allowed",
        criticality="error",
        check_func=check_funcs.is_not_null_and_is_in_list,
        column="status",
        check_func_kwargs={
            "allowed": ["PAID", "PENDING", "CANCELLED"]
        },
    ),
    DQDatasetRule(
        name="order_id_unique",
        criticality="error",
        check_func=check_funcs.is_unique,
        columns=["order_id"],
    ),
    DQRowRule(
        name="high_value_order_review",
        criticality="warn",
        check_func=check_funcs.is_in_range,
        column="amount",
        check_func_kwargs={"min_limit": 0, "max_limit": 1_000},
    ),
]

print(f"Defined {len(checks)} checks.")

## 6. Create the engine

The SDK client uses the current notebook's Databricks authentication context.

In [ ]:
workspace_client = WorkspaceClient()
dq_engine = DQEngine(workspace_client)

print("DQX engine is ready.")

## 7. Apply checks and keep all rows

`apply_checks` preserves the input rows and adds structured result columns. This view is ideal for investigation and reporting.

In [ ]:
checked_orders_df = dq_engine.apply_checks(orders_df, checks)

display(
    checked_orders_df.select(
        "order_id", "customer_id", "amount", "status",
        "_errors", "_warnings",
    )
)

## 8. Split valid and quarantine data

`apply_checks_and_split` returns two DataFrames:

- **valid**: records with no errors; warning-only records remain valid;
- **quarantine**: records with an error or warning, including diagnostic columns.

Therefore, a warning-only row can appear in both outputs. This is intentional.

In [ ]:
valid_orders_df, quarantine_orders_df = dq_engine.apply_checks_and_split(
    orders_df, checks
)

print("Input rows     :", orders_df.count())
print("Valid rows     :", valid_orders_df.count())
print("Quarantine rows:", quarantine_orders_df.count())

print("Valid output")
display(valid_orders_df.orderBy("order_id"))
print("Quarantine output")
display(quarantine_orders_df.orderBy("order_id"))

## 9. Flatten error details for analysis

Each item in `_errors` contains fields such as the rule name, function, message, affected columns, and runtime. Exploding the array produces a useful issue table.

In [ ]:
error_details_df = (
    quarantine_orders_df
    .select("order_id", F.explode_outer("_errors").alias("issue"))
    .where(F.col("issue").isNotNull())
    .select("order_id", "issue.name", "issue.message", "issue.columns")
)

display(error_details_df.orderBy("order_id", "name"))

## 10. Summarize failures by rule

A small aggregation makes the result suitable for a dashboard or alert.

In [ ]:
failure_summary_df = (
    error_details_df
    .groupBy("name")
    .agg(F.count("*").alias("failed_rows"))
    .orderBy(F.desc("failed_rows"), "name")
)

display(failure_summary_df)

## 11. Declarative checks with dictionaries

DQX also supports YAML, JSON, dictionaries, and Delta tables. Declarative rules are easier to store outside pipeline code. This small example checks the same business ideas using metadata.

In [ ]:
metadata_checks = [
    {
        "name": "customer_id_required_metadata",
        "criticality": "error",
        "check": {
            "function": "is_not_null_and_not_empty",
            "arguments": {"column": "customer_id", "trim_strings": True},
        },
    },
    {
        "name": "status_allowed_metadata",
        "criticality": "error",
        "check": {
            "function": "is_not_null_and_is_in_list",
            "arguments": {
                "column": "status",
                "allowed": ["PAID", "PENDING", "CANCELLED"],
            },
        },
    },
]

metadata_result_df = dq_engine.apply_checks_by_metadata(
    orders_df, metadata_checks
)
display(metadata_result_df.select("order_id", "_errors"))

## 12. Build a pipeline quality gate

A pipeline may continue with valid records while separately retaining quarantine records for investigation. A maximum tolerated error rate makes that decision explicit.

In [ ]:
def quality_gate(dataframe, rules, max_error_rate=0.50):
    valid_df, quarantine_df = dq_engine.apply_checks_and_split(dataframe, rules)

    total_rows = dataframe.count()
    error_rows = quarantine_df.where(F.size("_errors") > 0).count()
    error_rate = error_rows / total_rows if total_rows else 0.0

    if error_rate > max_error_rate:
        raise ValueError(
            f"Quality gate failed: {error_rows}/{total_rows} rows "
            f"have errors ({error_rate:.1%})"
        )

    return valid_df, quarantine_df, error_rate

try:
    trusted_df, rejected_df, rate = quality_gate(
        orders_df, checks, max_error_rate=0.25
    )
    print(f"Quality gate passed; error rate = {rate:.1%}")
except ValueError as error:
    print("Pipeline stopped as expected for this faulty sample.")
    print(error)

## 13. Optional: save outputs as Delta tables

The lesson avoids writes so it runs in any Free Edition workspace. In your own catalog/schema, the usual pattern is:

```python
valid_orders_df.write.mode("overwrite").saveAsTable("catalog.schema.orders_valid")
quarantine_orders_df.write.mode("overwrite").saveAsTable("catalog.schema.orders_quarantine")
```

Use append/merge semantics, permissions, retention, and sensitive-data controls appropriate to the real pipeline.

## 14. Beginner exercises

1. Correct the negative amount and rerun the split.
2. Change `high_value_order_review` from `warn` to `error`. Where does order 103 appear?
3. Add `REFUNDED` to the allowed status list.
4. Add a new null `customer_id` row and compare `_errors` with `_warnings`.
5. Replace `orders_df` with `spark.table("catalog.schema.orders")`.
6. Store the declarative checks as YAML after you are comfortable with the dictionary structure.

## 15. Recap

- DQX applies Spark-native quality checks to DataFrames.
- Row rules identify individual bad values; dataset rules can detect conditions such as duplicate keys.
- `error` controls quarantine; `warn` records an issue without removing the row from valid output.
- `apply_checks` annotates all data; `apply_checks_and_split` produces valid and quarantine DataFrames.
- Structured result columns explain exactly which rules failed.
- Apply related checks together in production to avoid repeated passes over the same dataset.

Official references: [DQX overview](https://databrickslabs.github.io/dqx/), [installation](https://databrickslabs.github.io/dqx/docs/installation/), [defining checks](https://databrickslabs.github.io/dqx/docs/guide/quality_checks_definition/), and [applying checks](https://databrickslabs.github.io/dqx/docs/guide/quality_checks_apply/).